# CareerPilot AI — 4. Generate Explanations

Takes the top matches from `top_matches.json` and the profile from `careerpilot.db`, and asks a local Ollama model to write a short, human-readable explanation of why each job fits and what skill to learn next.

This is generation only — plain text in, plain text out over HTTP to Ollama. No training happens here either.

**Requires:** `3_job_matching.ipynb` has been run (so `top_matches.json` exists), and Ollama is running with a model already pulled on this server (check with `ollama list` in a terminal — you can't pull new models here since `/models` is read-only on this shared box, so pick one from what's already available).


## Step 1 — Setup


In [1]:
import json
import sqlite3
import time
import requests

DB_PATH = "careerpilot.db"
TOP_MATCHES_PATH = "top_matches.json"
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen3.5:4b"   # confirmed available via `ollama list` on this server
OLLAMA_TIMEOUT = 180          # generous — shared box + cold model load can be slow
MAX_EXPLANATIONS = 3          # how many top matches to generate explanations for


## Step 2 — Warm up the model

The first call to a model on a shared box can be slow while it loads into memory. Running one small request first means the real explanation calls below aren't competing with that cold-start delay. If this cell takes 30–90 seconds the first time, that's expected — not stuck.


In [2]:
start = time.time()
warm_response = requests.post(OLLAMA_URL, json={
    "model": OLLAMA_MODEL,
    "prompt": "Say ready.",
    "stream": False,
}, timeout=OLLAMA_TIMEOUT)

print(f"Took {time.time() - start:.1f}s")
if warm_response.status_code != 200:
    print("Ollama error response:", warm_response.text)
warm_response.raise_for_status()
print(warm_response.json().get("response"))


Took 36.7s
ready


## Step 3 — Load the profile and top matches


In [3]:
def load_profile(profile_id: int) -> dict:
    conn = sqlite3.connect(DB_PATH)
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM career_profiles WHERE id = ?", (profile_id,))
        row = cur.fetchone()
        cols = [d[0] for d in cur.description]
    finally:
        conn.close()
    if row is None:
        raise ValueError(f"No profile with id {profile_id} in careerpilot.db")
    data = dict(zip(cols, row))
    for f in ["skills", "education", "experience", "organizations", "certifications"]:
        data[f] = json.loads(data[f]) if data[f] else []
    return data

with open(TOP_MATCHES_PATH, "r") as f:
    match_data = json.load(f)

profile = load_profile(match_data["profile_id"])
matches = match_data["matches"][:MAX_EXPLANATIONS]

print(f"Profile: {profile.get('name') or profile.get('filename')}")
print(f"Generating explanations for {len(matches)} matches...")


ValueError: No profile with id 6 in careerpilot.db

## Step 4 — Call Ollama for each match

If a call fails, the error response text is printed before the exception, so you can see exactly what Ollama said instead of a generic timeout traceback.


In [4]:
def generate_explanation(profile: dict, job: dict) -> str:
    profile_skills = ", ".join(profile.get("skills", []))
    job_skills = ", ".join(job.get("skills", []))

    prompt = (
        f"Candidate skills: {profile_skills}.\n"
        f"Job: {job['title']} at {job['company']}. Required skills: {job_skills}.\n"
        f"Job description: {job['description'][:600]}\n\n"
        "In 2 sentences: explain why this is a good match for the candidate, and "
        "name one specific skill from the required list the candidate is missing "
        "and should learn next. If no skills are missing, say so."
    )

    start = time.time()
    response = requests.post(OLLAMA_URL, json={
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
    }, timeout=OLLAMA_TIMEOUT)

    if response.status_code != 200:
        print("Ollama error response:", response.text)
    response.raise_for_status()

    elapsed = time.time() - start
    print(f"   (generated in {elapsed:.1f}s)")
    return response.json().get("response", "").strip()

results = []
for rank, job in enumerate(matches, start=1):
    print(f"{rank}. {job['title']} — {job['company']}  (score: {job['score']:.3f})")
    explanation = generate_explanation(profile, job)
    results.append({**job, "explanation": explanation})

    print(f"   Apply: {job.get('apply_link', 'link not available')}")
    print(f"   {explanation}")
    print()


1. Technical Project Manager — Ascendion  (score: 0.575)


ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=180)

## Step 5 — Save final results (this is your demo output)


In [ ]:
with open("final_recommendations.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved final_recommendations.json — this is the end-to-end pipeline output.")
